# Decomposição da Série Temporal

Nesta etapa será realizada a decomposição da série temporal de Trocas de Óleo da MecâniQA, buscando isolar os componentes de Tendência, Sazonalidade e Ruído.

Com base no Brainstorm da equipe, foi adotado:

- Modelo: Aditivo
- Periodicidade: 7 dias

## 1. Importação das bibliotecas

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose

## 2. Carregamento e preparação do dataset

In [ ]:
data_path = next((
    path for path in [
        Path("data/mecaniqa_dataset.xlsx"),
        Path("../data/mecaniqa_dataset.xlsx"),
    ]
    if path.exists()
), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset não encontrado. Abra o projeto pela pasta mecaniQA-salvador."
    )

df = pd.read_excel(data_path)
df.head()

In [ ]:
df["Data"] = pd.to_datetime(df["Data"])

df = df.sort_values("Data")
df.set_index("Data", inplace=True)

df.head()

## 3. Série temporal de Trocas de Óleo

In [ ]:
serie_trocas_oleo = df["Trocas_Oleo"].resample("D").sum(min_count=1)

serie_trocas_oleo.head()

## 4. Decomposição da série temporal

In [ ]:
# A decomposição exige uma série completa. A interpolação é usada apenas
# nesta análise descritiva; os baselines abaixo usam somente dados passados.
serie_decomposicao = serie_trocas_oleo.interpolate(method="time")

decomposicao = seasonal_decompose(
    serie_decomposicao,
    model="additive",
    period=7
)

decomposicao

## 5. Visualização dos componentes

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

axes[0].plot(decomposicao.observed, color="#1f77b4")
axes[0].set_title("Série Observada - Trocas de Óleo")
axes[0].set_ylabel("Trocas")

axes[1].plot(decomposicao.trend, color="#2ca02c")
axes[1].set_title("Tendência")
axes[1].set_ylabel("Tendência")

axes[2].plot(decomposicao.seasonal, color="#ff7f0e")
axes[2].set_title("Sazonalidade")
axes[2].set_ylabel("Sazonalidade")

axes[3].plot(decomposicao.resid, color="#d62728")
axes[3].set_title("Ruído / Resíduos")
axes[3].set_ylabel("Resíduos")
axes[3].set_xlabel("Data")

plt.tight_layout()
plt.show()

## 6. Modelos de baseline protegidos contra Data Leakage

### Decisões do brainstorm de 02/09

1. **Armadilha do futuro:** aplicamos `shift(1)` antes das previsões e das janelas móveis. Assim, a previsão do instante `t` começa no valor observado em `t-1` e nunca usa `t+1`. Valores ausentes no histórico são preenchidos com `ffill`, que também consulta apenas o passado.
2. **Complexidade x simplicidade:** cada baseline é uma função Python curta que recebe a série e devolve outra série indexada pelas mesmas datas. Essa estrutura direta facilita comparar o Naive e diferentes janelas sem introduzir classes desnecessárias.

O modelo Naive repete o valor do dia anterior. A média móvel usa os sete valores anteriores ao dia previsto.

In [ ]:
def modelo_naive(serie):
    return serie.shift(1)


def modelo_media_movel(serie, janela=7):
    return serie.shift(1).rolling(window=janela).mean()


# ffill preserva a ordem temporal: um vazio recebe o último valor conhecido.
serie_historica = serie_trocas_oleo.ffill()
previsao_naive = modelo_naive(serie_historica)
previsao_media_movel_7 = modelo_media_movel(serie_historica, janela=7)

previsoes_baseline = pd.DataFrame({
    "Real": serie_trocas_oleo,
    "Naive": previsao_naive,
    "Média móvel (7 dias)": previsao_media_movel_7,
})
previsoes_baseline.head(10)

### Comparação visual dos baselines

O recorte final deixa o deslocamento mais legível, enquanto os cálculos permanecem disponíveis para toda a série.

In [ ]:
ax = previsoes_baseline.tail(60).plot(figsize=(14, 6), linewidth=2)
ax.set_title("Trocas de óleo: valores reais e previsões baseline")
ax.set_xlabel("Data")
ax.set_ylabel("Quantidade de trocas de óleo")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Próxima etapa: atividade de 09/09

Os dois baselines estão prontos para receber a validação temporal. Na atividade de 09/09, aplicar `TimeSeriesSplit` e calcular MAE, RMSE e MAPE nas mesmas janelas de teste para os dois modelos. Ao calcular MAPE, tratar explicitamente datas cujo valor real seja zero.